In [30]:
import numpy as np
import pandas as pd
import pickle
import h5py
import math
import os

In [2]:
### LOADING ECFP4 FINGERPRINTS FROM HDF5 FILE ###

In [3]:
root = "."
H5_PATH  = os.path.join(root, "..", "processed", "enamine_REAL_characterization", "enamine_REAL_ECFP4.h5")

with h5py.File(H5_PATH, "r") as f:
    fps = f['fps'][:]
    ids = f['ids'][:]
    popc = f['popc'][:]

print("fps shape:", fps.shape, fps.dtype)
print("ids shape:", ids.shape)
print("popc shape:", popc.shape, popc.dtype)

def tanimoto_u64(a: np.ndarray, b: np.ndarray, pa: int, pb: int) -> float:
    inter = sum(int((x & y).bit_count()) for x, y in zip(a, b))
    denom = pa + pb - inter
    return inter / denom if denom else 1.0

def bitbound_possible(pa: int, pb: int, thr: float) -> bool:
    return min(pa, pb) / max(pa, pb) >= thr

fps shape: (9557694, 32) uint64
ids shape: (9557694,)
popc shape: (9557694,) uint16


In [4]:
### LOADING SUCCESS MOLECULES (IN CONFORMATION GENERATION) ###
PATH_TO_INFERRED_PROBS = os.path.join(root, "..", "processed", "unidock_docking", "inference_probs")
success_mols = np.array(pickle.load(open(os.path.join(PATH_TO_INFERRED_PROBS, "success_mols.pkl"), "rb")))

In [5]:
### LOAD MAPPINGS ###
df = pd.read_csv(os.path.join(root, "..", "processed", "enamine_REAL_characterization", "enamine_REAL.tsv"), sep='\t')
df['index'] = df.index
id_to_index = dict(zip(df['id'], df['index']))
index_to_id = dict(zip(df['index'], df['id']))

In [6]:
### Load pocket detection data ###
pocket_detection_data = pd.read_csv(os.path.join(root, "..", "processed", "pocket_detection_data.csv"))

In [8]:
for file, pocket_n in zip(pocket_detection_data['File name'], pocket_detection_data['Pocket number']):

    # Pocket label
    pocket = file.replace(".pdb", "") + f"_pocket_{pocket_n}"
    
    # Load inference probabilities
    probs = np.load(os.path.join(PATH_TO_INFERRED_PROBS, f"{pocket}_bin_01.npz"))['arr_0']
    inds_probs = np.argsort(probs)[::-1]

    # Sort molecules by inferred probability
    sorted_molecules = success_mols[inds_probs]

    # From sorted molecules, get their indices in the full dataset
    sorted_indices = np.array([id_to_index[mol_id] for mol_id in sorted_molecules])

    break

In [47]:
POPC8 = np.array([bin(i).count("1") for i in range(256)], dtype=np.uint8)

def any_sim_ge_thr(fp_cand, pa, leader_idx, leader_pc, fps, thr):
    if len(leader_idx) == 0:
        return False
    L = fps[leader_idx]  # shape (M, 32) uint64
    # Intersection: (M,32) uint64 -> view as bytes (M,256) -> popcount with POPC8 -> (M,)
    inter = POPC8[(L & fp_cand).view(np.uint8)].reshape(L.shape[0], 256).sum(axis=1).astype(np.int32)
    den   = pa + leader_pc - inter
    # inter/den >= thr  <=> inter >= thr*den
    return np.any(inter.astype(np.float64) >= thr * den)

In [51]:
THR = 0.70
K = 100000
selected_inds, selected_pc = [], []
buckets = [[] for _ in range(2049)]

for k, ind in enumerate(sorted_indices[:1000]):

    # If we already have K molecules, stop
    if len(selected_inds) >= K:
        break

    # Get fingerprint and popcount of the candidate molecule
    fp_cand = fps[ind]
    pc_cand = popc[ind]
    pc_cand = int(popc[ind])

    if not selected_inds:
        selected_inds.append(ind)
        selected_pc.append(pc_cand)
        buckets[pc_cand].append(0)       # this is the position in selected_inds
        continue

    # Calculate popcount window for viable comparisons
    low_bound = max(0, math.ceil(THR * pc_cand))
    high_bound = min(2048, math.floor(pc_cand / THR) if THR > 0 else 2048)

    # # Decide whether to keep the candidate
    # keep = True
    # for b in range(low_bound, high_bound + 1):
    #     for pos in buckets[b]:                     
    #         sel_ind = selected_inds[pos]
    #         fp_sel = fps[sel_ind]
    #         pc_sel = selected_pc[pos]
    #         # Calculate exact Tanimoto 
    #         if tanimoto_u64(fp_cand, fp_sel, pc_cand, pc_sel) >= THR:
    #             keep = False
    #             break
    #     if keep == False:
    #         break

    # Decide whether to keep the candidate (batch over eligible buckets)
    keep = True
    cand_positions = [pos for b in range(low_bound, high_bound + 1) for pos in buckets[b]]
    if cand_positions:
        leader_idx = np.array([selected_inds[pos] for pos in cand_positions], dtype=np.int64)
        leader_pc  = np.array([int(selected_pc[pos]) for pos in cand_positions], dtype=np.int32)
        if any_sim_ge_thr(fp_cand, pc_cand, leader_idx, leader_pc, fps, THR):
            keep = False

    # If we decided to keep the candidate, add it to the selected list
    if keep:
        pos = len(selected_inds)
        selected_inds.append(int(ind))
        selected_pc.append(int(pc_cand))
        buckets[int(pc_cand)].append(pos)

    if (k+1) % 100 == 0:
        print(f"Processed {k+1} molecules, selected {len(selected_inds)} so far.")
    

Processed 100 molecules, selected 100 so far.
Processed 200 molecules, selected 198 so far.
Processed 300 molecules, selected 297 so far.
Processed 400 molecules, selected 395 so far.
Processed 500 molecules, selected 495 so far.
Processed 600 molecules, selected 595 so far.
Processed 700 molecules, selected 694 so far.
Processed 800 molecules, selected 794 so far.
Processed 900 molecules, selected 892 so far.
Processed 1000 molecules, selected 991 so far.
